In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.sparse import csr_matrix

def simulate_scrna_seq(
    n_samples=10,
    n_genes=2500, 
    n_de_genes=200,
    cells_per_sample=500,
    mean_expression=100,
    desired_sparsity=0.95,
    n_recursive_steps=2,
    random_seed=666
):
    """
    Comprehensive scRNA-seq simulation with multimodal expression patterns
    
    Parameters:
    -----------
    n_samples: Number of samples (default: 10)
    n_genes: Number of genes (default: 2500)
    n_de_genes: Number of differentially expressed genes (default: 200)
    cells_per_sample: Number of cells per sample (default: 500)
    mean_expression: Average expression level (default: 100)
    desired_sparsity: Target sparsity level (default: 0.95)
    n_recursive_steps: Number of recursive scaling steps (default: 2)
    random_seed: Random seed for reproducibility (default: 666)
    """
    
    # Set random seed
    np.random.seed(random_seed)
    
    # Define sample names and group assignments
    sample_names = [f'Sample{i+1}' for i in range(n_samples)]
    sample_to_group = {}
    for i, sample_name in enumerate(sample_names):
        if i < 4:
            sample_to_group[sample_name] = 'Group0'
        else:
            sample_to_group[sample_name] = 'Group1'
    
    # Sample heterogeneity factors (0.9 to 1.1)
    sample_scaling = {name: np.random.uniform(0.9, 1.1) for name in sample_names}
    
    # Define cells per sample with slight variation
    cells_per_sample_dict = {
        name: np.random.poisson(cells_per_sample) for name in sample_names
    }
    
    # Total number of cells
    total_cells = sum(cells_per_sample_dict.values())
    
    # Generate base mean expression levels using Gamma distribution
    mu_base = np.random.gamma(shape=mean_expression/10, scale=10.0, size=n_genes)
    
    # Generate dispersion parameter theta for negative binomial
    theta = np.random.gamma(shape=2.0, scale=1.0, size=n_genes)
    
    # Define mean expression for each group
    mu_group = {
        'Group0': mu_base.copy(),
        'Group1': mu_base.copy()
    }
    
    # Randomly select DE genes
    de_gene_indices = np.random.choice(n_genes, size=n_de_genes, replace=False)
    de_gene_indices_up = de_gene_indices[:100]  # First 100 are upregulated
    de_gene_indices_down = de_gene_indices[100:]  # Next 100 are downregulated
    
    # Initialize data storage
    all_counts = []
    all_cell_ids = []
    all_sample_ids = []
    all_group_ids = []
    
    # Track DE gene scaling info for validation
    scaling_info = {
        'up_factors': [],
        'down_factors': [],
        'affected_cells': {}
    }
    
    # Simulate data for each sample
    for sample_idx, sample_name in enumerate(sample_names):
        group_id = sample_to_group[sample_name]
        n_cells = cells_per_sample_dict[sample_name]
        sample_scale = sample_scaling[sample_name]
        
        # Apply sample heterogeneity to mean expression
        sample_mu = mu_group[group_id] * sample_scale
        
        # Simulate counts using negative binomial distribution
        # For NB: mean = mu, variance = mu + mu²/theta
        counts = np.random.negative_binomial(
            n=theta,
            p=theta / (theta + sample_mu),
            size=(n_cells, n_genes)
        )
        
        # Ensure counts are integers and non-negative
        counts = np.maximum(counts, 0).astype(int)
        
        # Apply multimodal scaling for DE genes in Group 0
        if group_id == 'Group0':
            # Track which cells are affected
            affected_cells = np.ones(n_cells, dtype=bool)
            scaling_info['affected_cells'][sample_name] = []
            
            # Recursive scaling process
            for step in range(n_recursive_steps):
                # Select 50% of currently active cells
                n_active = np.sum(affected_cells)
                n_to_scale = max(1, int(n_active * 0.5))
                active_indices = np.where(affected_cells)[0]
                selected_indices = np.random.choice(active_indices, size=n_to_scale, replace=False)
                
                # Generate scaling factors for this step
                up_factors = np.random.uniform(1.1, 8.0, size=len(selected_indices))
                down_factors = np.random.uniform(0.1, 0.9, size=len(selected_indices))
                
                # Apply scaling to selected cells
                for idx, cell_idx in enumerate(selected_indices):
                    counts[cell_idx, de_gene_indices_up] = np.floor(
                        counts[cell_idx, de_gene_indices_up] * up_factors[idx]
                    ).astype(int)
                    counts[cell_idx, de_gene_indices_down] = np.floor(
                        counts[cell_idx, de_gene_indices_down] * down_factors[idx]
                    ).astype(int)
                
                # Update affected cells for next iteration
                affected_cells = np.zeros(n_cells, dtype=bool)
                affected_cells[selected_indices] = True
                
                # Store scaling info
                scaling_info['affected_cells'][sample_name].append(selected_indices.tolist())
                scaling_info['up_factors'].extend(up_factors.tolist())
                scaling_info['down_factors'].extend(down_factors.tolist())
        
        # Apply expression-dependent dropout mechanism
        # Calculate dropout probabilities
        log_counts = np.log1p(counts)  # log(count + 1)
        max_log_count = np.max(log_counts)
        if max_log_count > 0:
            dropout_prob = (1 - log_counts / max_log_count) * desired_sparsity
        else:
            dropout_prob = np.ones_like(counts) * desired_sparsity
        
        # Apply dropout
        random_values = np.random.rand(*counts.shape)
        counts[random_values < dropout_prob] = 0
        
        # Collect data
        all_counts.append(counts)
        
        # Generate cell IDs
        cell_ids = [f'{sample_name}_Cell{i+1}' for i in range(n_cells)]
        all_cell_ids.extend(cell_ids)
        all_sample_ids.extend([sample_name] * n_cells)
        all_group_ids.extend([group_id] * n_cells)
    
    # Combine all counts
    counts_matrix = np.vstack(all_counts)
    
    # Create gene names
    gene_names = [f'Gene_{i+1}' for i in range(n_genes)]
    
    # Mark DE genes
    gene_info = pd.DataFrame({
        'gene_id': gene_names,
        'is_de': False,
        'de_direction': 'none'
    })
    gene_info.loc[de_gene_indices_up, 'is_de'] = True
    gene_info.loc[de_gene_indices_up, 'de_direction'] = 'up'
    gene_info.loc[de_gene_indices_down, 'is_de'] = True
    gene_info.loc[de_gene_indices_down, 'de_direction'] = 'down'
    
    # Create cell metadata
    obs_data = pd.DataFrame({
        'cell_id': all_cell_ids,
        'sample': all_sample_ids,
        'group': all_group_ids
    })
    obs_data.index = obs_data['cell_id']
    
    # Create AnnData object
    adata = ad.AnnData(
        X=csr_matrix(counts_matrix),
        obs=obs_data,
        var=gene_info.set_index('gene_id')
    )
    
    # Add simulation parameters to uns
    adata.uns['simulation_params'] = {
        'n_samples': n_samples,
        'n_genes': n_genes,
        'n_de_genes': n_de_genes,
        'cells_per_sample': cells_per_sample,
        'mean_expression': mean_expression,
        'desired_sparsity': desired_sparsity,
        'n_recursive_steps': n_recursive_steps,
        'distribution': 'negative_binomial',
        'random_seed': random_seed
    }
    
    # Calculate and store summary statistics
    actual_sparsity = (counts_matrix == 0).sum() / counts_matrix.size
    
    # Calculate mean expression for DE genes by group
    group0_mask = np.array(all_group_ids) == 'Group0'
    group1_mask = np.array(all_group_ids) == 'Group1'
    
    stats = {
        'actual_sparsity': actual_sparsity,
        'total_cells': len(all_cell_ids),
        'cells_per_group': {
            'Group0': group0_mask.sum(),
            'Group1': group1_mask.sum()
        },
        'de_gene_stats': {
            'up_genes': {
                'mean_group0': counts_matrix[group0_mask][:, de_gene_indices_up].mean(),
                'mean_group1': counts_matrix[group1_mask][:, de_gene_indices_up].mean(),
                'fold_change': counts_matrix[group0_mask][:, de_gene_indices_up].mean() / 
                              (counts_matrix[group1_mask][:, de_gene_indices_up].mean() + 0.001)
            },
            'down_genes': {
                'mean_group0': counts_matrix[group0_mask][:, de_gene_indices_down].mean(),
                'mean_group1': counts_matrix[group1_mask][:, de_gene_indices_down].mean(),
                'fold_change': counts_matrix[group0_mask][:, de_gene_indices_down].mean() / 
                              (counts_matrix[group1_mask][:, de_gene_indices_down].mean() + 0.001)
            }
        }
    }
    
    adata.uns['simulation_stats'] = stats
    
    # Print summary
    print("=== scRNA-seq Simulation Summary ===")
    print(f"Total cells: {stats['total_cells']}")
    print(f"Cells in Group0: {stats['cells_per_group']['Group0']}")
    print(f"Cells in Group1: {stats['cells_per_group']['Group1']}")
    print(f"Number of genes: {n_genes}")
    print(f"Number of DE genes: {n_de_genes} (100 up, 100 down)")
    print(f"Actual sparsity: {actual_sparsity:.4f}")
    print(f"\nDE genes (upregulated):")
    print(f"  Mean in Group0: {stats['de_gene_stats']['up_genes']['mean_group0']:.4f}")
    print(f"  Mean in Group1: {stats['de_gene_stats']['up_genes']['mean_group1']:.4f}")
    print(f"  Fold change: {stats['de_gene_stats']['up_genes']['fold_change']:.4f}")
    print(f"\nDE genes (downregulated):")
    print(f"  Mean in Group0: {stats['de_gene_stats']['down_genes']['mean_group0']:.4f}")
    print(f"  Mean in Group1: {stats['de_gene_stats']['down_genes']['mean_group1']:.4f}")
    print(f"  Fold change: {stats['de_gene_stats']['down_genes']['fold_change']:.4f}")
    
    return adata, scaling_info

# Run simulation
if __name__ == "__main__":
    adata_scrna, scaling_info = simulate_scrna_seq()
    
    # Save data
    adata_scrna.write_h5ad('simulated_scrna_data.h5ad')
    print("\nData saved to: simulated_scrna_data.h5ad")